<a href="https://colab.research.google.com/github/kokila83/AgenticAIMani_SampleMiniProject/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import getpass
from openai import OpenAI
import numpy as np

# 1. Enter your Azure AI Foundry API key
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client pointing to your Azure endpoint
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

# 3. Reference your exact deployment name
EMBED_MODEL = "text-embedding-3-small"

# Sample texts
text1 = "I love machine learning"
text2 = "AI and ML are fascinating"
text3 = "I ate pizza for dinner"

# Generate embeddings
res1 = client.embeddings.create(model=EMBED_MODEL, input=text1)
res2 = client.embeddings.create(model=EMBED_MODEL, input=text2)
res3 = client.embeddings.create(model=EMBED_MODEL, input=text3)

emb1 = res1.data[0].embedding
emb2 = res2.data[0].embedding
emb3 = res3.data[0].embedding

print(f"Embedding dimensions: {len(emb1)}")

# Cosine similarity function
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_1_2 = cosine_similarity(emb1, emb2)
sim_1_3 = cosine_similarity(emb1, emb3)

print(f"Similarity (ML texts): {sim_1_2:.3f}")    # High score
print(f"Similarity (ML vs pizza): {sim_1_3:.3f}") # Low score

Enter your Azure AI Foundry API key: ··········
Embedding dimensions: 1536
Similarity (ML texts): 0.623
Similarity (ML vs pizza): 0.166


In [5]:
# Ex 2: Simple RAG without Vector DB

import getpass
from openai import OpenAI
import numpy as np

# 1. Setup client cleanly
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

# Knowledge base
documents = [
    "Our company was founded in 2020 in Bangalore.",
    "We offer three products: ChatBot Pro, DataAnalyzer, AIWriter.",
    "ChatBot Pro costs $99/month for 10,000 messages.",
    "Our CEO is Rahul Sharma, who worked at Google.",
]

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-mini"

# Embed text helper
def embed_text(text):
    result = client.embeddings.create(model=EMBED_MODEL, input=text)
    return result.data[0].embedding

# Generate document embeddings
doc_embeddings = [embed_text(doc) for doc in documents]

# Simple RAG function
def simple_rag(question, top_k=2):
    q_embedding = embed_text(question)

    def cosine_sim(a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

    similarities = [cosine_sim(q_embedding, doc_emb) for doc_emb in doc_embeddings]
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    relevant_docs = [documents[i] for i in top_indices]

    # Safe console printing (emojis are fine inside python print, but keep them out of API payloads)
    print(f"Retrieved: {relevant_docs}")

    prompt = f"""Based on these documents:
{'\n'.join(relevant_docs)}

Question: {question}
Answer based only on documents:"""

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print(simple_rag("How much does ChatBot Pro cost?"))

Enter your Azure AI Foundry API key: ··········
Retrieved: ['ChatBot Pro costs $99/month for 10,000 messages.', 'We offer three products: ChatBot Pro, DataAnalyzer, AIWriter.']
ChatBot Pro costs $99 per month for 10,000 messages.


In [7]:
#Ex 3: RAG with ChromaDB (Vector Database)
!pip install chromadb
import getpass
from openai import OpenAI
import chromadb

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure client pointing to your Azure AI Foundry endpoint
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

# 3. Initialize ChromaDB client and collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="company_docs")

# Knowledge base
documents = [
    "Python is a programming language by Guido van Rossum.",
    "Machine learning is a subset of AI.",
    "TensorFlow and PyTorch are popular ML frameworks.",
    "GPT stands for Generative Pre-trained Transformer.",
]

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-mini"

# Helper function to get embeddings via Azure AI Foundry
def get_embedding(text):
    result = client.embeddings.create(model=EMBED_MODEL, input=text)
    return result.data[0].embedding

# Add documents and their embeddings to ChromaDB
for i, doc in enumerate(documents):
    collection.add(
        documents=[doc],
        embeddings=[get_embedding(doc)],
        ids=[f"doc_{i}"]
    )

print(f"Added {collection.count()} documents to ChromaDB")

# RAG Query function
def rag_query(question, n_results=2):
    # Query ChromaDB using vector embeddings
    results = collection.query(
        query_embeddings=[get_embedding(question)],
        n_results=n_results
    )
    relevant_docs = results['documents'][0]

    # Emojis stay in print statements (not sent in API payloads)
    print(f"Retrieved docs: {relevant_docs}")

    prompt = f"""Context:
{'\n'.join(relevant_docs)}

Question: {question}
Answer:"""

    # Fixed missing parenthesis from original snippet
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Test the RAG setup
print(rag_query("What is GPT?"))
print("-" * 50)
print(rag_query("Who created Python?"))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

In [8]:
# Ex 4: Agentic RAG
import getpass
import json
from openai import OpenAI

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client pointing to your Azure AI Foundry endpoint
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

CHAT_MODEL = "gpt-4.1-mini"

# Mock knowledge base
knowledge_base = {
    "pricing": "Basic: $10/mo, Pro: $50/mo, Enterprise: Custom",
    "features": "Chat, Analytics, Integrations, API access",
    "support": "Email: 24h, Phone: Business hours, Enterprise: 24/7",
    "company": "Founded 2020, HQ in Bangalore, 500 employees"
}

def search_knowledge(query):
    query_lower = query.lower()
    for key, value in knowledge_base.items():
        if key in query_lower:
            return f"Found: {value}"
    return "No relevant information found"

def agentic_rag(user_question):
    # Prompt instructing agent to respond in strictly valid JSON format
    decision_prompt = f"""User question: {user_question}

You have access to a company knowledge base.
Should you search or answer directly?

Respond strictly in valid JSON format:
{{"action": "search", "query": "..."}} OR {{"action": "answer", "response": "..."}}"""

    # 3. Decision Step: Agent decides whether retrieval is necessary
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": decision_prompt}]
    )

    try:
        raw_text = response.choices[0].message.content.strip()
        # Clean potential markdown formatting from LLM output
        if raw_text.startswith("```json"):
            raw_text = raw_text.replace("```json", "").replace("```", "").strip()

        decision = json.loads(raw_text)

        if decision.get("action") == "search":
            search_query = decision.get("query", user_question)
            retrieved = search_knowledge(search_query)
            print(f"🔍 Agent searched: {search_query}")
            print(f"📚 Retrieved: {retrieved}")

            # Answer Step using retrieved context
            answer_prompt = f"""Context: {retrieved}
Question: {user_question}
Answer:"""
            final = client.chat.completions.create(
                model=CHAT_MODEL,
                messages=[{"role": "user", "content": answer_prompt}]
            )
            return final.choices[0].message.content
        else:
            print("💡 Agent answered directly")
            return decision.get("response", raw_text)

    except json.JSONDecodeError:
        return response.choices[0].message.content

# 4. Test Agentic Decision Routing
if __name__ == "__main__":
    print("--- Test 1: Search Required ---")
    print("Response:", agentic_rag("What are your pricing plans?"))
    print("\n" + "="*50 + "\n")
    print("--- Test 2: Direct Answer ---")
    print("Response:", agentic_rag("What's 2+2?"))

Enter your Azure AI Foundry API key: ··········
--- Test 1: Search Required ---
🔍 Agent searched: pricing plans
📚 Retrieved: Found: Basic: $10/mo, Pro: $50/mo, Enterprise: Custom
Response: Our pricing plans are as follows:
- Basic: $10 per month
- Pro: $50 per month
- Enterprise: Custom pricing tailored to your needs


--- Test 2: Direct Answer ---
💡 Agent answered directly
Response: 2+2 equals 4.
